# apcorr · el flujo de la primaria, paso a paso

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`

Los otros notebooks `debug` auditan **una etapa** cada uno. Éste sigue **una sola cantidad** —el flujo de la estrella— desde el cubo hasta el producto, y en cada paso enseña **por qué función se multiplica**.

Existe por un número: el cociente entre el producto y una apertura sobre el cubo, dividido por la `apcorr`, **debería ser plano** y varía un factor **1.19**. Mientras eso no se cierre, la forma del continuo no es citable.

Poco texto: **una figura por paso**.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_aqui = Path.cwd()
ROOT = next(p for p in (_aqui, *_aqui.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Del **config resuelto de cada etapa**, nunca copiadas como literales: la etapa rellena defaults que el run no escribe.


In [ ]:
from musepipe.stages.stage_x03_psffit import stage_x03_config_from_run
from musepipe.stages.stage_e01_psf import stage_e01_config_from_run
from musepipe.growth_curve import resolve_flux_convention as _res_conv

X03 = stage_x03_config_from_run(RUN_ID, project_root=ROOT)
E01 = stage_e01_config_from_run(RUN_ID, project_root=ROOT)
STAR_RADIUS_PX = float(X03.get('x03_star_radius_px', 20.0))
COMP_RADIUS_PX = float(X03.get('x03_comp_radius_px', 12.0))
APCORR_MODE    = X03.get('x03_aperture_correction', 'auto')
GROWTH, CONVENCION = _res_conv(X03, SD, knob='x03_flux_convention')

# Radio de normalización del modelo de PSF: el '1' de la convención
# `normrad`. Todo el problema vive entre este radio y el total.
R_NORM = float(E01.get('e01_norm_radius_px', 25.0))
R_GRANDE = 32.0    # apertura grande: recoge casi todo el halo medible

# 1 de cada N canales. El ajuste es independiente por canal, así que
# los muestreados salen idénticos a los de la cadena completa.
PASO_CANALES = 5

print(f'convención de flujo: {CONVENCION}'
      f"  ({'con' if GROWTH else 'SIN'} curva de crecimiento empírica)")
print(f'radios: normalización {R_NORM:.0f} px · apertura grande {R_GRANDE:.0f} px'
      f' | 1 de cada {PASO_CANALES} canales')


## 2 · La escalera de entradas, con su fecha

Un run puede tener piezas de vintages distintos. Si algo aquí está fechado antes que su entrada, lo que salga describe un estado que ya no existe.


In [ ]:
import datetime as _dt

ESCALERA = [
    ('B1/B2  cubo de entrada', SD / 'stage02_xcorr_cube_stack.fits'),
    ('A3     transmisión telúrica', SD / 'TELLURIC_TRANS.fits'),
    ('C1     modelo de PSF', SD / 'psf_model.json'),
    ('A2     curva de crecimiento', SD / 'growth_curve_qc.json'),
    ('C4     primaria sin calibrar', SD / 'spec_psffit_star.fits'),
    ('D2     primaria calibrada', SD / 'spec_calibrated_psffit_star.fits'),
]
for etiqueta, ruta in ESCALERA:
    if ruta.exists():
        cuando = _dt.datetime.fromtimestamp(ruta.stat().st_mtime)
        print(f'  {etiqueta:28s} {cuando:%Y-%m-%d %H:%M}  {ruta.name}')
    else:
        print(f'  {etiqueta:28s} {"AUSENTE":16s}  {ruta.name}')


## 3 · Las funciones copiadas de `musepipe`

Incluye `factor_at_wavelengths`: **es la función bajo sospecha**, así que viaja en la copia y se puede editar aquí sin tocar la cadena.

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `_npix_eff` — de `musepipe/extraction/aperture.py`
- `aperture_spectrum` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`
- `covariance_factor_for_npix` — de `musepipe/extraction/optimal.py`
- `estimate_variance_cube` — de `musepipe/extraction/optimal.py`
- `PsfFitCubeResult` — de `musepipe/extraction/psffit.py`
- `fit_region_mask` — de `musepipe/extraction/psffit.py`
- `psf_pair_design` — de `musepipe/extraction/psffit.py`
- `_correlation` — de `musepipe/extraction/psffit.py`
- `_fit_one_channel` — de `musepipe/extraction/psffit.py`
- `fit_psffit_cube` — de `musepipe/extraction/psffit.py`
- `control_psffit_spectra` — de `musepipe/extraction/psffit.py`
- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`
- `interior_bump` — de `musepipe/growth_curve.py`
- `interior_dip` — de `musepipe/growth_curve.py`
- `polynomial_misfit` — de `musepipe/growth_curve.py`
- `factor_at_wavelengths` — de `musepipe/growth_curve.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from dataclasses import dataclass
from musepipe.parallel import run_channel_chunks
from musepipe.psf import evaluate_psf_model
from typing import Sequence
import math
import numpy as np
import warnings
from dataclasses import dataclass
# `evaluate_psf_model` (C1) se importa arriba: es lo que C1 entrega,
# no lo que se audita aquí.

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8
MAX_INTERIOR_BUMP = 0.03
DEFAULT_FACTOR_METHOD = "pchip"


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        if size < 1 or size % 2 == 0:
            raise ValueError(
                "box aperture size must be an odd positive integer, got "
                f"size={size}. An even size has no integer-centred box: "
                f"`half = size // 2` would silently return the {2 * (size // 2) + 1}x"
                f"{2 * (size // 2) + 1} one under the wrong label."
            )
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def _npix_eff(cube_zyx: np.ndarray, weights: np.ndarray) -> np.ndarray:
    valid = np.isfinite(cube_zyx) & (weights[None, :, :] > 0)
    sumw = np.sum(weights[None, :, :] * valid, axis=(1, 2))
    sumw2 = np.sum((weights[None, :, :] ** 2) * valid, axis=(1, 2))
    out = np.full(cube_zyx.shape[0], np.nan, dtype=np.float64)
    good = sumw2 > 0
    out[good] = (sumw[good] ** 2) / sumw2[good]
    return out


def aperture_spectrum(cube_zyx, center_yx, aperture: dict) -> tuple[np.ndarray, np.ndarray]:
    """Return weighted-sum spectrum and per-channel effective pixel count."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    weights = aperture_weights(ny, nx, center_yx, aperture)
    weighted = cube * weights[None, :, :]
    with np.errstate(invalid="ignore"):
        flux = np.nansum(weighted, axis=(1, 2)).astype(np.float64)
    npix_eff = _npix_eff(cube, weights)
    flux[~np.isfinite(npix_eff)] = np.nan
    return flux, npix_eff


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
    growth_curve=None,
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model.

    By default the PSF is normalized to 1 inside ``norm_radius_px`` (25 px =
    0.63" in NFM), so the recovered "total flux" is really *the flux inside that
    radius*. Measured on the A2-size cube, 40-55% of the modelled light lies
    outside it, and the missing factor is chromatic (~2.5 blue, ~1.9 red), so it
    does not cancel -- it tilts the continuum.

    Passing ``growth_curve`` (A2's ``growth_curve`` QC block) multiplies the
    correction by the empirically measured ``F_total / F(<=norm_radius)`` and
    switches the convention to genuine total flux. A2 measures it; the caller
    decides -- see ``x01_flux_convention``.
    """

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    apcorr = (1.0 / fractions).astype(np.float64)
    if growth_curve:
        # Import ABSOLUTO y dentro de la funcion: esta funcion se COPIA
        # literalmente dentro de los notebooks de `debug/`, donde un import
        # relativo (`from ..growth_curve`) revienta con ImportError por no
        # haber paquete padre. El absoluto funciona en los dos sitios.
        from musepipe.growth_curve import factor_at_wavelengths

        factor = np.asarray(factor_at_wavelengths(growth_curve, wave), dtype=np.float64)
        if not np.all(np.isfinite(factor)) or np.any(factor <= 0):
            raise RuntimeError("Growth-curve total-flux factor is not finite and positive.")
        return apcorr * factor, "psf_growth_curve+empirical_total", norm_radius
    return apcorr, "psf_growth_curve", norm_radius


def covariance_factor_for_npix(npix_eff, covariance_factor_box3=1.0):
    """Linearly interpolate covariance inflation between one pixel and box3."""

    vals = np.asarray(npix_eff, dtype=np.float64)
    box3 = float(covariance_factor_box3)
    if not np.isfinite(box3) or box3 <= 0:
        box3 = 1.0
    t = np.clip((vals - 1.0) / 8.0, 0.0, 1.0)
    return 1.0 + t * (box3 - 1.0)


def estimate_variance_cube(cube_zyx):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    out = np.empty_like(cube, dtype=np.float64)
    for i in range(cube.shape[0]):
        sigma = robust_sigma(cube[i])
        if not np.isfinite(sigma) or sigma <= 0:
            sigma = 1.0
        out[i] = sigma**2
    return out


@dataclass(frozen=True)
class PsfFitCubeResult:
    coeffs: np.ndarray
    covariance: np.ndarray
    chi2r: np.ndarray
    condition_number: np.ndarray
    rho_ab: np.ndarray
    rho_bc: np.ndarray
    npix: np.ndarray
    npix_eff_comp: np.ndarray
    residual_cube: np.ndarray
    model_cube: np.ndarray
    fit_mask: np.ndarray


def fit_region_mask(shape, star_yx, comp_yx, *, star_radius_px=20.0, comp_radius_px=12.0):
    ny, nx = map(int, shape)
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    sy, sx = map(float, star_yx)
    cy, cx = map(float, comp_yx)
    return ((yy - sy) ** 2 + (xx - sx) ** 2 <= float(star_radius_px) ** 2) | (
        (yy - cy) ** 2 + (xx - cx) ** 2 <= float(comp_radius_px) ** 2
    )


def psf_pair_design(shape, wave_A, star_yx, comp_yx, psf_model):
    yy, xx = np.indices(shape, dtype=np.float64)
    p_star = evaluate_psf_model(
        psf_model,
        float(wave_A),
        yy - float(star_yx[0]),
        xx - float(star_yx[1]),
    )
    p_comp = evaluate_psf_model(
        psf_model,
        float(wave_A),
        yy - float(comp_yx[0]),
        xx - float(comp_yx[1]),
    )
    y0 = 0.5 * (float(star_yx[0]) + float(comp_yx[0]))
    x0 = 0.5 * (float(star_yx[1]) + float(comp_yx[1]))
    scale = max(float(np.hypot(float(comp_yx[0]) - float(star_yx[0]), float(comp_yx[1]) - float(star_yx[1]))), 1.0)
    y_scaled = (yy - y0) / scale
    x_scaled = (xx - x0) / scale
    return np.stack([p_star, p_comp, np.ones(shape), y_scaled, x_scaled], axis=-1)


def _correlation(cov, i, j):
    denom = float(cov[i, i] * cov[j, j])
    if not np.isfinite(denom) or denom <= 0:
        return np.nan
    return float(cov[i, j] / np.sqrt(denom))


def _fit_one_channel(data_2d, variance_2d, design_3d, fit_mask):
    data = np.asarray(data_2d, dtype=np.float64)
    variance = np.asarray(variance_2d, dtype=np.float64)
    design = np.asarray(design_3d, dtype=np.float64)
    valid = np.asarray(fit_mask, dtype=bool) & np.isfinite(data) & np.isfinite(variance) & (variance > 0)
    valid &= np.all(np.isfinite(design), axis=-1)
    n = int(np.count_nonzero(valid))
    p = design.shape[-1]
    if n <= p:
        coeff = np.full(p, np.nan, dtype=np.float64)
        cov = np.full((p, p), np.nan, dtype=np.float64)
        return coeff, cov, np.nan, np.inf, np.nan, np.nan, n, np.nan, np.full_like(data, np.nan), np.full_like(data, np.nan)

    a = design[valid].reshape(n, p)
    y = data[valid]
    var = variance[valid]
    sw = 1.0 / np.sqrt(var)
    aw = a * sw[:, None]
    yw = y * sw
    coeff, *_ = np.linalg.lstsq(aw, yw, rcond=None)
    normal = aw.T @ aw
    cov = np.linalg.pinv(normal)
    model = np.tensordot(design, coeff, axes=([-1], [0]))
    resid = data - model
    dof = max(1, n - p)
    chi2r = float(np.nansum((resid[valid] ** 2) / var) / dof)

    col_norm = np.linalg.norm(aw, axis=0)
    safe = col_norm > 0
    if np.all(safe):
        condition = float(np.linalg.cond(aw / col_norm[None, :]))
    else:
        condition = np.inf
    rho_ab = _correlation(cov, 0, 1)
    rho_bc_vals = [_correlation(cov, 1, j) for j in (2, 3, 4)]
    rho_bc = float(np.nanmax(np.abs(rho_bc_vals))) if np.any(np.isfinite(rho_bc_vals)) else np.nan
    pcomp = a[:, 1]
    npix_eff = np.nan
    if np.sum(pcomp**2) > 0:
        npix_eff = float((np.sum(pcomp) ** 2) / np.sum(pcomp**2))
    return coeff, cov, chi2r, condition, rho_ab, rho_bc, n, npix_eff, model, resid


def fit_psffit_cube(
    cube_zyx,
    variance_zyx,
    wave_A,
    star_yx,
    comp_yx,
    psf_model,
    *,
    star_radius_px=20.0,
    comp_radius_px=12.0,
    n_jobs=1,
) -> PsfFitCubeResult:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    variance = np.asarray(variance_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")
    if wave.ndim != 1 or wave.size != cube.shape[0]:
        raise ValueError("wave_A must match cube spectral length.")

    nz, ny, nx = cube.shape
    fit_mask = fit_region_mask((ny, nx), star_yx, comp_yx, star_radius_px=star_radius_px, comp_radius_px=comp_radius_px)
    coeffs = np.full((nz, 5), np.nan, dtype=np.float64)
    cov = np.full((nz, 5, 5), np.nan, dtype=np.float64)
    chi2r = np.full(nz, np.nan, dtype=np.float64)
    cond = np.full(nz, np.nan, dtype=np.float64)
    rho_ab = np.full(nz, np.nan, dtype=np.float64)
    rho_bc = np.full(nz, np.nan, dtype=np.float64)
    npix = np.zeros(nz, dtype=np.int32)
    npix_eff = np.full(nz, np.nan, dtype=np.float64)
    model_cube = np.full_like(cube, np.nan, dtype=np.float64)
    residual_cube = np.full_like(cube, np.nan, dtype=np.float64)

    def _fit_range(z0, z1):
        # Per-channel work identical to the serial loop; disjoint output slots.
        for z in range(z0, z1):
            design = psf_pair_design((ny, nx), wave[z], star_yx, comp_yx, psf_model)
            row = _fit_one_channel(cube[z], variance[z], design, fit_mask)
            coeffs[z], cov[z], chi2r[z], cond[z], rho_ab[z], rho_bc[z], npix[z], npix_eff[z], model_cube[z], residual_cube[z] = row

    run_channel_chunks(_fit_range, nz, n_jobs=n_jobs)
    return PsfFitCubeResult(
        coeffs=coeffs,
        covariance=cov,
        chi2r=chi2r,
        condition_number=cond,
        rho_ab=rho_ab,
        rho_bc=rho_bc,
        npix=npix,
        npix_eff_comp=npix_eff,
        residual_cube=residual_cube,
        model_cube=model_cube,
        fit_mask=fit_mask,
    )


def control_psffit_spectra(
    cube_zyx,
    variance_zyx,
    wave_A,
    star_yx,
    comp_yx,
    psf_model,
    *,
    star_radius_px=20.0,
    comp_radius_px=12.0,
    n_controls=8,
    exclude_angle_deg=25.0,
    n_jobs=1,
) -> tuple[list[tuple[int, int]], np.ndarray, np.ndarray]:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    _, ny, nx = cube.shape
    controls = same_radius_control_positions(
        comp_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(np.ceil(comp_radius_px)) + 1,
    )
    star_specs = []
    comp_specs = []
    for center in controls:
        fit = fit_psffit_cube(
            cube,
            variance_zyx,
            wave_A,
            star_yx,
            center,
            psf_model,
            star_radius_px=star_radius_px,
            comp_radius_px=comp_radius_px,
            n_jobs=n_jobs,
        )
        star_specs.append(fit.coeffs[:, 0])
        comp_specs.append(fit.coeffs[:, 1])
    if not controls:
        return controls, np.empty((0, cube.shape[0])), np.empty((0, cube.shape[0]))
    return controls, np.asarray(star_specs, dtype=np.float64), np.asarray(comp_specs, dtype=np.float64)


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


def interior_bump(values):
    """Cuanto sube un maximo INTERIOR por encima del mayor de los dos extremos.

    0.0 si la curva es monotona (el maximo cae en un extremo). Es la medida que
    separa una joroba real de un repunte de borde: no mira signos de la
    derivada, mira tamano.
    """

    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size < 3:
        return 0.0
    borde = max(float(vals[0]), float(vals[-1]))
    if borde <= 0:
        return 0.0
    return max(0.0, float(np.max(vals)) / borde - 1.0)


def interior_dip(values):
    """Cuanto sube la curva desde un minimo INTERIOR hasta su extremo final.

    El espejo de `interior_bump`, y hace falta porque `interior_bump` **no ve
    este caso**: una curva que baja, toca fondo dentro del rango y vuelve a
    subir tiene su maximo en un extremo, asi que `interior_bump` devuelve 0.0 y
    el diagnostico la declara «monotona». Sobre `ROXs12b_realigned` la parabola
    de `factor_at_wavelengths` toca fondo en 7978 A y repunta un 3.15% hasta
    9350 A, mientras la curva medida entre 8487 y 9062 A sube 0.37%: el
    repunte lo fabrica el ajuste, y `interior_bump` daba 0.00%.

    0.0 si la curva es monotona (el minimo cae en un extremo).
    """

    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size < 3:
        return 0.0
    i = int(np.argmin(vals))
    if i in (0, vals.size - 1):
        return 0.0
    fondo = float(vals[i])
    if fondo <= 0:
        return 0.0
    # Solo cuenta lo que sube DESPUES del minimo: el tramo de bajada anterior
    # es justo lo que la curva debe hacer.
    return max(0.0, float(np.max(vals[i:])) / fondo - 1.0)


def polynomial_misfit(growth_qc, *, degree=2):
    """`(rms, pico_a_pico)` del ajuste polinomico contra las bandas medidas.

    `factor_at_wavelengths` interpola 8 bandas con un polinomio de grado bajo
    para no meter escalones en el continuo, pero nadie comprobaba cuanto se
    aparta ese polinomio de lo que se midio. En el run canonico vale 1.03% rms
    y 2.82% pico a pico, y es lo que produce el repunte rojo que mide
    `interior_dip`.
    """

    bands = (growth_qc or {}).get("bands") or []
    if len(bands) < degree + 2:
        return None, None
    x = np.asarray([b["wave_A"] for b in bands], dtype=np.float64)
    y = np.asarray([b["ratio_total_over_normrad"] for b in bands], dtype=np.float64)
    fit = np.polyval(np.polyfit(x, y, int(degree)), x)
    rel = fit / y - 1.0
    return float(np.std(rel)), float(np.max(rel) - np.min(rel))


def factor_at_wavelengths(growth_qc, wave_A, *, method=None, degree=2,
                          require_monotonic=True):
    """Interpola el factor por banda a un eje de longitudes de onda.

    Las bandas son 8 puntos de una curva suave, y hace falta una interpolacion
    que no meta escalones en el continuo. Hay dos:

    * ``"pchip"`` (**por defecto desde 2026-08-11**) -- cubica monotona a trozos
      (Fritsch-Carlson). Pasa **por** las bandas y no inventa curvatura entre
      ellas: si el dato baja, ella baja. Fuera del rango de bandas extrapola
      **en recta** con la pendiente del extremo, no con la cubica, que se
      dispara.
    * ``"poly2"`` -- el ajuste polinomico historico. Se conserva para reproducir
      runs congelados; para eso hace falta ademas `require_monotonic=False`
      (ver abajo).

    **Por que cambio el defecto.** Una parabola tiene curvatura constante: no
    puede bajar deprisa y luego aplanarse, que es justo lo que hace esta curva.
    En `ROXs12b_realigned` las 8 bandas caen de 1.684 a 1.460 y se aplanan
    (el ultimo punto sube un 0.37%, ruido), y la parabola les pone el **vertice
    dentro del rango**, en 7978 A, repuntando un **3.15%** hasta 9350 A. Como
    la apcorr multiplica, el continuo por encima de ~8000 A salia
    **sobre-corregido en los seis metodos del bloque C** a la vez. La parabola
    ni siquiera reproducia las bandas: 1.03% rms, 2.82% pico a pico
    (`polynomial_misfit`). Con `pchip` el repunte cae a lo que digan las bandas
    y el residuo contra ellas es cero por construccion.

    **Se exige monotonia** (2026-08-07, ampliado 2026-08-11). El factor es
    `F_total / F(<=r_norm)`, o sea el inverso de una fraccion encerrada: con AO
    el Strehl empeora hacia el azul, mas luz se va al halo y la correccion tiene
    que **caer** del azul al rojo. Se miran los dos modos de fallo:

    * un **maximo interior** (`interior_bump`) -- la parabola de las bandas del
      cubo de 200 px lo tenia en ~7400 A y deformaba la pendiente del continuo
      de TODO el bloque C un ~47%. Bandas mal medidas: se re-miden en un campo
      mayor, no se interpolan.
    * un **minimo interior con repunte** (`interior_dip`) -- invisible para
      `interior_bump`, porque el maximo cae en un extremo y devolvia 0.00%. Es
      el caso de arriba, y con `poly2` lo fabrica el ajuste, no el dato.
    """

    method = str(method or DEFAULT_FACTOR_METHOD).lower()
    if method not in {"pchip", "poly2", "poly"}:
        raise ValueError(f"Unknown method={method!r}; expected 'pchip' or 'poly2'.")
    bands = (growth_qc or {}).get("bands") or []
    minimo = 2 if method == "pchip" else int(degree) + 1
    if len(bands) < minimo:
        raise ValueError(f"Need at least {minimo} bands to fit the factor, got {len(bands)}.")
    x = np.asarray([b["wave_A"] for b in bands], dtype=np.float64)
    y = np.asarray([b["ratio_total_over_normrad"] for b in bands], dtype=np.float64)
    orden = np.argsort(x)           # PCHIP exige x creciente; el QC no lo promete
    x, y = x[orden], y[orden]
    wave = np.asarray(wave_A, dtype=np.float64)
    if method == "pchip":
        from scipy.interpolate import PchipInterpolator

        spline = PchipInterpolator(x, y, extrapolate=False)
        out = spline(wave)
        # Fuera de las bandas, RECTA con la pendiente del extremo. La cubica
        # extrapolada se curva sin dato que la sujete, y aqui se extrapola de
        # verdad: el eje del cubo llega a 4750 A y la banda mas azul esta en
        # 5037. Una recta al menos no puede dar la vuelta.
        pend = spline.derivative()
        azul, rojo = wave < x[0], wave > x[-1]
        out[azul] = y[0] + (wave[azul] - x[0]) * float(pend(x[0]))
        out[rojo] = y[-1] + (wave[rojo] - x[-1]) * float(pend(x[-1]))
    else:
        out = np.polyval(np.polyfit(x, y, int(degree)), wave)
    if require_monotonic:
        # Se juzga sobre el eje pedido, que es donde se aplica: un polinomio
        # puede tener el vertice fuera del rango de las bandas y dentro del
        # rango del espectro.
        #
        # Con TOLERANCIA, no por signo: un test estricto marca tambien un
        # repunte del 1% en el borde por ruido numerico o por extrapolacion del
        # ajuste. Lo que importa es el TAMANO de la joroba: cuanto sube el
        # maximo interior por encima del mayor de los dos extremos. En el caso
        # que motivo esto valia 11%; en una curva sana vale 0.
        if np.isfinite(out).sum() > 2:
            joroba = interior_bump(out)
            if joroba > MAX_INTERIOR_BUMP:
                pico = float(wave[np.nanargmax(out)])
                raise RuntimeError(
                    f"The growth-curve factor is not monotonic in wavelength: it bumps {100 * joroba:.1f}% "
                    f"above its endpoints, peaking at {pico:.0f} A. It is the inverse of an "
                    "enclosed fraction: under AO it must "
                    "decrease from blue to red. Non-monotonic bands mean the halo/sky fit is "
                    "degenerate — re-measure on a wider field with "
                    "`scripts/measure_growth_curve.py --cube <cubo ancho> --write-run-product`. "
                    f"bands={np.array2string(y, precision=3)}")
            # El otro modo de fallo, que hasta 2026-08-11 pasaba entero: bajar,
            # tocar fondo DENTRO del rango y repuntar. El maximo queda en un
            # extremo, asi que `interior_bump` da 0.00% y esto se colaba.
            hundido = interior_dip(out)
            if hundido > MAX_INTERIOR_BUMP:
                fondo = float(wave[np.nanargmin(out)])
                culpa = (
                    "the degree-2 fit manufactures it (a parabola cannot fall and then "
                    "flatten); use method='pchip', the default"
                    if method != "pchip" else
                    "the measured bands themselves turn up; re-measure on a wider field with "
                    "`scripts/measure_growth_curve.py --cube <cubo ancho> --write-run-product`")
                raise RuntimeError(
                    f"The growth-curve factor is not monotonic in wavelength: it bottoms out at "
                    f"{fondo:.0f} A and rebounds {100 * hundido:.1f}% to the red end. It is the "
                    "inverse of an enclosed fraction: under AO it must decrease from blue to red. "
                    f"Here {culpa}. bands={np.array2string(y, precision=3)}")
    return out


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "5553fdf997f8",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:_npix_eff": "32ecf15dcce9",
    "musepipe/extraction/aperture.py:aperture_spectrum": "214c68e30b47",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "a2d1719a4ffc",
    "musepipe/extraction/optimal.py:covariance_factor_for_npix": "bfff5c5c63d8",
    "musepipe/extraction/optimal.py:estimate_variance_cube": "52dd9aee1ded",
    "musepipe/extraction/psffit.py:PsfFitCubeResult": "149f25f484cb",
    "musepipe/extraction/psffit.py:fit_region_mask": "90cdb2410161",
    "musepipe/extraction/psffit.py:psf_pair_design": "c61762fea651",
    "musepipe/extraction/psffit.py:_correlation": "4e2f284568ab",
    "musepipe/extraction/psffit.py:_fit_one_channel": "5fe11d3ec8fb",
    "musepipe/extraction/psffit.py:fit_psffit_cube": "706a881da6bc",
    "musepipe/extraction/psffit.py:control_psffit_spectra": "713d8f221856",
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/growth_curve.py:interior_bump": "46e7526e1b93",
    "musepipe/growth_curve.py:interior_dip": "211848110880",
    "musepipe/growth_curve.py:polynomial_misfit": "22c13393ee97",
    "musepipe/growth_curve.py:factor_at_wavelengths": "612f3d307441",
    "musepipe/extraction/aperture.py:FLAG_BAD_WINDOW": "bcdec9eb9722",
    "musepipe/extraction/aperture.py:FLAG_SKYLINE": "d4bfcfc18628",
    "musepipe/extraction/aperture.py:FLAG_INTERPOLATED": "5fb1fecc67d5",
    "musepipe/extraction/aperture.py:FLAG_CLIPPED": "00efdddd962b",
    "musepipe/growth_curve.py:MAX_INTERIOR_BUMP": "250487d17461",
    "musepipe/growth_curve.py:DEFAULT_FACTOR_METHOD": "32d9a3adcf3a"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} APCORR')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · Paso 0 — lo que el DRS ya hizo

El punto de partida **no es el crudo**: un crudo de MUSE son 24 pixtables por IFU y no tiene espectro. Lo primero que existe como espectro es el cubo, y para entonces esorex ya aplicó bias, flat, calibración en λ y **la calibración de flujo con la estrella estándar**. Eso fija la unidad; lo que viene después solo cambia la forma.


In [ ]:
QC_A1 = nb.load_qc_optional('stages/stage00r_qc.json', RUN_ID) or {}
QC_A4 = nb.load_qc_optional('stages/stage00q_qc.json', RUN_ID) or {}
from musepipe.io import resolve_bunit

CUBO = SD / 'stage02_xcorr_cube_stack.fits'
with fits.open(CUBO) as _h:
    CUBE_FULL = np.asarray(_h['CUBES'].data, dtype=float)
    WAVE_FULL = np.asarray(_h['WAVELENGTH'].data, dtype=float)
    STAT_FULL = np.asarray(_h['STAT'].data, dtype=float) if 'STAT' in _h else None
    _bunit_stack = str(_h[0].header.get('BUNIT', '')
                       or _h['CUBES'].header.get('BUNIT', '')) or None
if CUBE_FULL.ndim == 4:
    CUBE_FULL = CUBE_FULL[0]
if STAT_FULL is not None and STAT_FULL.ndim == 4:
    STAT_FULL = STAT_FULL[0]
UNIDAD = resolve_bunit(X03, stack_bunit=_bunit_stack) or 'sin unidad declarada'
print('cubo   :', CUBE_FULL.shape, f'({WAVE_FULL[0]:.0f}-{WAVE_FULL[-1]:.0f} Å)')
print('unidad :', UNIDAD)
_m3 = (QC_A4.get('m3_flux') or {})
if _m3.get('flux_factor') is not None:
    print(f"A4/M3  : escala contra Gaia {_m3['band']} ="
          f" {float(_m3['flux_factor']):.4f}  (estado {_m3.get('status')})")
    print('         se MIDE y se declara; no se aplica al flujo (ver §10).')
print('los pasos siguientes NO cambian la unidad: solo multiplican por'
      ' funciones de λ.')


## 6 · Paso 1 — el dato: dos aperturas y su cociente

Suma de apertura sobre el mismo cubo, a `R_NORM` y a `R_GRANDE`. **Sin sustraer fondo**, igual que la §10 de `D2_primary_star_debug`, para que los números sean comparables entre notebooks: lo que hay fuera del núcleo es halo de la propia estrella, no cielo (esorex ya lo restó).

El cociente de las dos curvas es **el cromatismo real del dato**, medido sin ningún modelo. Es la vara con la que se mide todo lo demás.


In [ ]:
QC_B3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
STAR_YX = tuple(float(v) for v in QC_B3['primary']['pos_yx'])
COMP_YX = tuple(float(v) for v in QC_B3['companion']['pos_yx'])
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))

CANALES = np.arange(0, WAVE_FULL.size, PASO_CANALES)
CUBE = CUBE_FULL[CANALES]
WAVE = WAVE_FULL[CANALES]
STAT = None if STAT_FULL is None else STAT_FULL[CANALES]

def suma_apertura(radio):
    """Suma simple dentro de un círculo, sin pesos ni fondo."""
    return extract_aperture_spectrum(CUBE, STAR_YX, float(radio))

F_NORM = suma_apertura(R_NORM)
F_GRANDE = suma_apertura(R_GRANDE)
with np.errstate(invalid='ignore', divide='ignore'):
    CROMA_DATO = F_GRANDE / F_NORM
_fin = np.isfinite(CROMA_DATO)
print(f'F(<={R_GRANDE:.0f}) / F(<={R_NORM:.0f}) medido:'
      f' {float(np.nanmin(CROMA_DATO[_fin])):.3f} .. {float(np.nanmax(CROMA_DATO[_fin])):.3f}'
      f'  (mediana {float(np.nanmedian(CROMA_DATO)):.3f})')
# Precalculado a propósito: una expresión de f-string NO puede partirse
# entre dos literales concatenados.
RANGO_CROMA_DATO = (float(np.nanpercentile(CROMA_DATO[_fin], 98))
                    / float(np.nanpercentile(CROMA_DATO[_fin], 2)))
print(f'varía un {100 * (RANGO_CROMA_DATO - 1):.1f}% de punta a punta'
      '  <- ESTE es el cromatismo que hay que reproducir')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.4), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(WAVE, F_NORM, lw=0.7, color='tab:blue', label=f'apertura r={R_NORM:.0f} px')
ax1.plot(WAVE, F_GRANDE, lw=0.7, color='tab:orange', label=f'apertura r={R_GRANDE:.0f} px')
ax1.set_ylabel(f'suma [{UNIDAD}]'); ax1.legend(fontsize=8)
ax1.set_title('paso 1 · el dato, sin ningún modelo', fontsize=9)
ax2.plot(WAVE, CROMA_DATO, lw=0.8, color='k')
ax2.set_ylabel('F(grande)/F(norm)', fontsize=8); ax2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 7 · Paso 2 — el telúrico

A3 mide la transmisión sobre esta misma estrella y **divide el cubo por ella**. O sea que el cubo del paso 1 ya la lleva dentro: aquí se enseña la función aplicada, que actúa solo dentro de sus bandas y no puede explicar una deformación de banda ancha.


In [ ]:
# El accesor de la cadena, no una lectura a mano: la curva no vive
# bajo `stages/` sino donde diga `products.transmission` del QC de A3,
# y buscarla por nombre daba «AUSENTE» con el fichero delante.
from musepipe.telluric_lines import measured_transmission

_medida = measured_transmission(RUN_ID, project_root=ROOT)
TRANS = None
if _medida is not None:
    TRANS = np.interp(WAVE, np.asarray(_medida['wave_A'], dtype=float),
                      np.asarray(_medida['transmission'], dtype=float))
    print('curva de A3:', str(_medida['source']).split('/')[-1])
if TRANS is None:
    # No es que falte: puede que A3 haya decidido NO corregir, que es
    # un resultado y no una ausencia. Se dice cuál de las dos es.
    _qc_a3 = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID) or {}
    _dec = _qc_a3.get('decision') or {}
    if _dec:
        print(f"A3 decidió NO corregir (telluric_applied ={_dec.get('telluric_applied')}):"
              ' la función de este paso es la IDENTIDAD.')
        for _banda, _prof in (_dec.get('depth_pct_by_band') or {}).items():
            print(f'   {_banda:10s} profundidad medida {float(_prof):5.2f}%')
        print('   -> el paso 2 no puede explicar nada de la deformación en este objeto.')
    else:
        print('este run no tiene QC de A3: no se puede decir nada del paso 2.')
else:
    _dentro = TRANS < 0.995
    print(f'transmisión: mínimo {float(np.nanmin(TRANS)):.3f},'
          f' {100 * _dentro.mean():.1f}% de los canales por debajo de 0.995')
    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.plot(WAVE, TRANS, lw=0.8, color='tab:green')
    ax.axhline(1.0, color='0.6', lw=0.7, ls=':')
    ax.set_ylabel('T(λ)', fontsize=8); ax.set_xlabel('λ [Å]')
    ax.set_title('paso 2 · la función telúrica: actúa SOLO en sus bandas', fontsize=9)
    fig.tight_layout(); plt.show()


## 8 · Paso 3 — el psffit: la amplitud del modelo

C4 no suma píxeles: ajusta **dos PSF normalizadas** (primaria y compañero) más un plano, y se queda con la amplitud. Como la PSF va normalizada a 1 dentro de `R_NORM`, esa amplitud **es el flujo dentro de ese radio** — la convención `normrad`.


In [ ]:
VARIANZA = (estimate_variance_cube(CUBE) if STAT is None
            else np.asarray(STAT, dtype=float)
            * float(X03.get('x03_stat_factor_spaxel', 1.0) or 1.0))
AJUSTE = fit_psffit_cube(CUBE, VARIANZA, WAVE, STAR_YX, COMP_YX, PSF_MODEL,
                         star_radius_px=STAR_RADIUS_PX,
                         comp_radius_px=COMP_RADIUS_PX, n_jobs=1)
F_PSFFIT = AJUSTE.coeffs[:, 0]
print(f'{WAVE.size} canales ajustados · χ²ᵣ mediano'
      f' {float(np.nanmedian(AJUSTE.chi2r)):.3f}')
with np.errstate(invalid='ignore', divide='ignore'):
    _r = F_PSFFIT / F_NORM
print(f'psffit / apertura r={R_NORM:.0f}: mediana {float(np.nanmedian(_r)):.3f}'
      '  (1.0 = el modelo reparte la luz como el dato dentro del radio)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.4), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(WAVE, F_NORM, lw=0.7, color='tab:blue', label=f'apertura r={R_NORM:.0f} px')
ax1.plot(WAVE, F_PSFFIT, lw=0.7, color='tab:red', label='psffit (amplitud del modelo)')
ax1.set_ylabel(f'[{UNIDAD}]'); ax1.legend(fontsize=8)
ax1.set_title('paso 3 · las dos maneras de medir DENTRO del mismo radio', fontsize=9)
ax2.plot(WAVE, _r, lw=0.8, color='tab:purple')
ax2.axhline(1.0, color='0.6', lw=0.7, ls=':')
ax2.set_ylabel('psffit / apertura', fontsize=8); ax2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 9 · Paso 4 — × `apcorr`

Aquí es donde se pasa de «flujo dentro de `R_NORM`» a «flujo total». Para la primaria por psffit la `apcorr` nominal es **1** (spec C4 §3.4), así que lo único que multiplica es **el factor empírico de la curva de crecimiento**, `F_total/F(≤R_NORM)`, interpolado de 8 bandas medidas.


In [ ]:
# `require_monotonic=False` a propósito: si la curva de este objeto está
# rota, la cadena PARA — y un notebook de diagnóstico tiene que poder
# dibujar justo esa curva. El veredicto del guardia se pide aparte y se
# imprime, que es más útil que morirse aquí.
APCORR = (np.ones(WAVE.size) if GROWTH is None else
          np.asarray(factor_at_wavelengths(GROWTH, WAVE, require_monotonic=False),
                     dtype=float))
F_TOTAL = F_PSFFIT * APCORR
GUARDIA_OK = True
if GROWTH is not None:
    try:
        factor_at_wavelengths(GROWTH, WAVE)
    except RuntimeError as _err:
        GUARDIA_OK = False
        print('LA CADENA PARA CON ESTA CURVA:')
        print('  ', str(_err)[:200])
        print('  -> para ESTE objeto, el paso 4 no es auditable: hay que'
             ' re-medir la curva antes de creerse nada de abajo.\n')
if GROWTH is None:
    print('este run no lleva curva de crecimiento: apcorr = 1 y el paso 4 no hace nada.')
else:
    _xb = np.array([b['wave_A'] for b in GROWTH['bands']], dtype=float)
    _yb = np.array([b['ratio_total_over_normrad'] for b in GROWTH['bands']], dtype=float)
    _o = np.argsort(_xb); _xb, _yb = _xb[_o], _yb[_o]
    print(f'factor: {float(APCORR[0]):.4f} (azul) .. {float(APCORR[-1]):.4f} (rojo)')
    print(f'joroba {100 * interior_bump(APCORR):.2f}% · repunte'
          f' {100 * interior_dip(APCORR):.2f}% · umbral'
          f' {100 * MAX_INTERIOR_BUMP:.0f}%')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.4), sharex=True,
                                   gridspec_kw={'height_ratios': [1, 2]})
    ax1.plot(WAVE, APCORR, lw=1.2, color='tab:red', label='apcorr aplicada')
    ax1.plot(_xb, _yb, 'ko', ms=4, label='las 8 bandas medidas')
    ax1.set_ylabel('factor', fontsize=8); ax1.legend(fontsize=8)
    ax1.set_title('paso 4 · la función que multiplica', fontsize=9)
    ax2.plot(WAVE, F_PSFFIT, lw=0.7, color='0.6', label='antes (normrad)')
    ax2.plot(WAVE, F_TOTAL, lw=0.7, color='k', label='después (total)')
    ax2.set_ylabel(f'[{UNIDAD}]'); ax2.set_xlabel('λ [Å]'); ax2.legend(fontsize=8)
    fig.tight_layout(); plt.show()


## 10 · Paso 5 — D2

Para la primaria, D2 **no toca el flujo**: solo desplaza el eje λ con lo que midió A4 sobre líneas de cielo. La escala de flujo se mide contra Gaia y, si sale consistente con 1, se **declara** en vez de aplicarse. La celda lo comprueba, no lo supone.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

C4_PROD = SpectrumProduct.read(SD / 'spec_psffit_star.fits')
D2_PROD = SpectrumProduct.read(SD / 'spec_calibrated_psffit_star.fits')
_fc4 = np.asarray(C4_PROD.flux, float); _fd2 = np.asarray(D2_PROD.flux, float)
_m = np.isfinite(_fc4) & np.isfinite(_fd2)
_dlam = float(np.nanmax(np.abs(np.asarray(D2_PROD.wave_A, float)
                               - np.asarray(C4_PROD.wave_A, float))))
print(f'flujo C4 -> D2: {"IDÉNTICO" if np.array_equal(_fc4[_m], _fd2[_m]) else "CAMBIA"}'
      f'  |  eje λ desplazado {_dlam:.4f} Å')
print('o sea: toda la FORMA del espectro entregado se decide en los pasos 3 y 4.')


## 11 · La cascada, de un vistazo

Todo lo anterior normalizado en la misma banda. Si la `apcorr` fuera correcta, las curvas de «apertura grande» y «psffit × apcorr» **coincidirían**: las dos dicen ser el flujo total de la misma estrella.


In [ ]:
BANDA_REF = (7750.0, 7860.0)

def _norm_banda(valores):
    _msk = ((WAVE >= BANDA_REF[0]) & (WAVE <= BANDA_REF[1])
            & np.isfinite(valores))
    return valores / float(np.nanmedian(valores[_msk])) if _msk.any() else valores * np.nan

PASOS = [('1 · apertura r=%.0f (normrad)' % R_NORM, F_NORM, 'tab:blue'),
         ('1 · apertura r=%.0f (casi total)' % R_GRANDE, F_GRANDE, 'tab:orange'),
         ('3 · psffit (normrad)', F_PSFFIT, 'tab:red'),
         ('4 · psffit × apcorr (total)', F_TOTAL, 'k')]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.5, 6.2), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
for etiqueta, valores, color in PASOS:
    ax1.plot(WAVE, _norm_banda(valores), lw=0.9, color=color, label=etiqueta)
ax1.axvspan(*BANDA_REF, color='0.9', zorder=0)
ax1.set_ylabel('normalizado en la banda gris'); ax1.legend(fontsize=8)
ax1.set_title('la cascada: si la apcorr fuese correcta, la naranja y la negra'
              ' coincidirían', fontsize=9)
with np.errstate(invalid='ignore', divide='ignore'):
    DESAJUSTE = _norm_banda(F_TOTAL) / _norm_banda(F_GRANDE)
ax2.plot(WAVE, DESAJUSTE, lw=0.8, color='tab:purple')
ax2.axhline(1.0, color='0.6', lw=0.7, ls=':')
ax2.set_ylabel('total(modelo) / casi-total(dato)', fontsize=8)
ax2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()
_f = np.isfinite(DESAJUSTE)
RANGO_DESAJUSTE = (float(np.nanpercentile(DESAJUSTE[_f], 98))
                   / float(np.nanpercentile(DESAJUSTE[_f], 2)))
print(f'desajuste de punta a punta: {100 * (RANGO_DESAJUSTE - 1):.0f}%')

# El MISMO número que imprime la §10 de `D2_primary_star_debug`, para
# que los dos notebooks se puedan contrastar: allí el cociente se
# divide además por la apcorr, y lo que queda es lo que la corrección
# NO explica.
with np.errstate(invalid='ignore', divide='ignore'):
    SIN_EXPLICAR = DESAJUSTE / _norm_banda(APCORR)
_f2b = np.isfinite(SIN_EXPLICAR) & (WAVE > 4800) & (WAVE < 9300)
RANGO_SIN_EXPLICAR = (float(np.nanpercentile(SIN_EXPLICAR[_f2b], 98))
                      / float(np.nanpercentile(SIN_EXPLICAR[_f2b], 2)))
print(f'y dividiendo además por la apcorr: queda un'
      f' {100 * (RANGO_SIN_EXPLICAR - 1):.0f}% SIN explicar'
      '   <- el número de la §10 de D2')


## 12 · Dónde entra la deformación

El cromatismo entre `R_NORM` y `R_GRANDE`, por dos caminos: **medido en el dato** (§6) y **predicho por el modelo de PSF de C1**. Si el modelo dice mucho menos que el dato, su halo es demasiado poco cromático y el psffit hereda ese error entero.


In [ ]:
_ap_grande = {'kind': 'circle', 'radius_px': float(R_GRANDE),
              'name': f'star_r{R_GRANDE:g}'}
_ap_norm = {'kind': 'circle', 'radius_px': float(R_NORM),
            'name': f'star_r{R_NORM:g}'}
# `aperture_correction_from_psf` devuelve 1/F(dentro): el cociente de
# las dos correcciones ES el cromatismo que predice el modelo.
_c_grande, _, _ = aperture_correction_from_psf(
    WAVE, _ap_grande, PSF_MODEL, center_yx=STAR_YX,
    correction_mode=APCORR_MODE, growth_curve=None)
_c_norm, _, _ = aperture_correction_from_psf(
    WAVE, _ap_norm, PSF_MODEL, center_yx=STAR_YX,
    correction_mode=APCORR_MODE, growth_curve=None)
CROMA_MODELO = _c_norm / _c_grande

def _rango(valores):
    _f2 = np.isfinite(valores)
    return (float(np.nanpercentile(valores[_f2], 98))
            / float(np.nanpercentile(valores[_f2], 2)))

print(f'cromatismo entre r={R_NORM:.0f} y r={R_GRANDE:.0f} px, de punta a punta:')
print(f'   medido en el dato : factor {_rango(CROMA_DATO):.3f}')
print(f'   dicho por el modelo: factor {_rango(CROMA_MODELO):.3f}')
print(f'   -> el modelo se queda {100 * (_rango(CROMA_DATO) / _rango(CROMA_MODELO) - 1):.0f}%'
      ' corto de cromatismo')

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(WAVE, CROMA_DATO / float(np.nanmedian(CROMA_DATO)), lw=0.9, color='k',
        label='medido en el dato')
ax.plot(WAVE, CROMA_MODELO / float(np.nanmedian(CROMA_MODELO)), lw=1.2,
        color='tab:red', label='predicho por el modelo de PSF (C1)')
ax.set_ylabel('cromatismo (normalizado)', fontsize=8); ax.set_xlabel('λ [Å]')
ax.set_title('la misma cantidad, por dos caminos', fontsize=9)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 13 · La corrección que sale del propio dato

`F(≤R_GRANDE) / F(≤R_NORM)` **no necesita modelo ni extrapolación**: es una división de dos sumas. No es el factor a total —hasta `R_GRANDE` sigue faltando halo— pero su **forma en λ** es la que la apcorr debería tener, y se puede comparar con la que lleva puesta.

Aquí no se decide nada: se deja el número al lado del otro.


In [ ]:
if GROWTH is None:
    print('sin curva de crecimiento: no hay apcorr que comparar.')
else:
    _emp = CROMA_DATO / float(np.nanmedian(CROMA_DATO))
    _apl = APCORR / float(np.nanmedian(APCORR))
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(WAVE, _emp, lw=0.9, color='k', label='forma medida en el dato')
    ax.plot(WAVE, _apl, lw=1.4, color='tab:red', label='forma de la apcorr aplicada')
    ax.set_ylabel('normalizado a su mediana', fontsize=8); ax.set_xlabel('λ [Å]')
    ax.set_title('§13 · la forma que debería tener, contra la que tiene', fontsize=9)
    ax.legend(fontsize=8); fig.tight_layout(); plt.show()
    with np.errstate(invalid='ignore', divide='ignore'):
        _coc = _emp / _apl
    _f3 = np.isfinite(_coc)
    RANGO_FORMA = (float(np.nanpercentile(_coc[_f3], 98))
                   / float(np.nanpercentile(_coc[_f3], 2)))
    print(f'discrepancia de forma: {100 * (RANGO_FORMA - 1):.0f}% de punta a punta')
    print('Las dos rutas para cerrarlo, y las dos son decisión científica:')
    print('  a) imponer la forma medida y dejar el modelo de C1 solo para el núcleo;')
    print('  b) rehacer el modelo de C1 para que su halo sea tan cromático como el dato.')


## 14 · El psffit salta donde C1 no ajustó

Hasta aquí la cascada trata el psffit como una caja negra. Esta sección la abre, porque el cociente **psffit / apertura `R_NORM`** —las dos maneras de medir dentro del *mismo* radio, §8— no baja suave: cae **en bloques** y vuelve a subir.

El ancho del bloque no es del cielo, es del código. `_evaluate_psfao` no evalúa la PSF en λ sino en `w_eff = round(λ / psfao_wave_bin_A) * psfao_wave_bin_A`, y sus parámetros salen de **interpolar** `param_table`, que solo tiene los bins que C1 ajustó de verdad (100 Å). Cuando la rejilla es más fina que esos bins —los 50 Å que se usaban cuando el documento no la declaraba— la mitad de los canales cae entre dos, y ahí nace el escalón.

La pregunta que lo decide: ¿el cociente depende de lo lejos que caiga `w_eff` del bin más cercano?


In [ ]:
SNAP_A = float(PSF_MODEL.get('psfao_wave_bin_A', 50.0))
NODOS = np.asarray((PSF_MODEL.get('param_table') or {}).get('lambda_A', []),
                   dtype=float)
# `np.round` y el `round` de `psf.py` empatan a par igual: mismo w_eff.
W_EFF = (np.round(WAVE / SNAP_A) * SNAP_A) if SNAP_A > 0 else WAVE.copy()
with np.errstate(invalid='ignore', divide='ignore'):
    RATIO_PSF = F_PSFFIT / F_NORM
# El notch del láser AO deja canales en blanco. Sin este filtro el «salto»
# mayor del cubo es el borde del notch, que no es un escalón del modelo.
VALIDO = (np.isfinite(RATIO_PSF)
          & (F_NORM > 0.5 * float(np.nanmedian(F_NORM))))
print(f'snap del modelo: {SNAP_A:.0f} Å · {NODOS.size} bins ajustados por C1'
      f' · {int(VALIDO.sum())}/{WAVE.size} canales válidos')

_, BLOQUE = np.unique(W_EFF, return_inverse=True)
N_BLOQUES = int(BLOQUE.max()) + 1
MED_BLOQUE = np.full(N_BLOQUES, np.nan)
for _b in range(N_BLOQUES):
    _m = (BLOQUE == _b) & VALIDO
    if _m.sum() >= 3:
        MED_BLOQUE[_b] = float(np.nanmedian(RATIO_PSF[_m]))
LAM_BLOQUE = np.array([float(np.nanmean(W_EFF[BLOQUE == _b]))
                       for _b in range(N_BLOQUES)])
# Salto de cada bloque respecto al anterior, solo entre bloques CONTIGUOS
# (los huecos del notch no son saltos, son ausencias).
SALTO = np.full(N_BLOQUES, np.nan)
_cont = np.isclose(np.diff(LAM_BLOQUE), SNAP_A)
with np.errstate(invalid='ignore', divide='ignore'):
    SALTO[1:][_cont] = (np.diff(MED_BLOQUE) / MED_BLOQUE[:-1])[_cont]
_fs = np.isfinite(SALTO)
print(f'salto entre bloques contiguos: |mediana|'
      f' {100 * float(np.nanmedian(np.abs(SALTO[_fs]))):.2f}%'
      f' · máximo {100 * float(np.nanmax(np.abs(SALTO[_fs]))):.1f}%')

if NODOS.size:
    DIST_NODO = np.abs(W_EFF[:, None] - NODOS[None, :]).min(axis=1)
    print('\npsffit/apertura según dónde cae w_eff:')
    for _lo, _hi, _etq in ((0.0, 25.0, 'encima de un bin de C1'),
                           (25.0, 75.0, 'a medio camino entre dos'),
                           (75.0, 200.0, 'a uno o dos bins'),
                           (200.0, np.inf, 'dentro de un hueco')):
        _m = VALIDO & (DIST_NODO >= _lo) & (DIST_NODO < _hi)
        if _m.sum() >= 20:
            print(f'   {_etq:26s} n={int(_m.sum()):5d}'
                  f'   {float(np.nanmedian(RATIO_PSF[_m])):.4f}')
    print('   -> si baja de arriba abajo, el daño no lo hace el ajuste de C1:'
          ' lo hace interpolar entre sus bins.')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.5, 5.6), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
for _n in NODOS:
    ax1.axvline(_n, color='0.85', lw=0.5, zorder=0)
ax1.plot(WAVE[VALIDO], RATIO_PSF[VALIDO], lw=0.5, color='0.55')
ax1.step(LAM_BLOQUE, MED_BLOQUE, where='mid', lw=1.2, color='k')
ax1.set_ylabel(f'psffit / apertura r={R_NORM:.0f}', fontsize=8)
ax1.set_title('§14 · gris vertical = los bins que C1 ajustó de verdad;'
              f' negro = mediana por bloque de {SNAP_A:.0f} Å', fontsize=9)
ax2.step(LAM_BLOQUE, 100 * SALTO, where='mid', lw=0.9, color='tab:purple')
ax2.axhline(0.0, color='0.6', lw=0.7, ls=':')
ax2.set_ylabel('salto [%]', fontsize=8); ax2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 15 · Los dos saltos mayores, retratados

Un escalón del cociente puede ser del dato o del modelo. Aquí se ve de quién: para los dos bloques con mayor salto se enseña, **antes y después**, el canal (mediana del bloque), el área que suma la apertura, el modelo del psffit y el residuo — y la resta de los tres.

Si el dato apenas cambia y el modelo cambia mucho, el escalón es nuestro.


In [ ]:
from matplotlib.colors import LogNorm

_orden = np.argsort(-np.abs(np.nan_to_num(SALTO)))
SALTOS_ELEGIDOS = []
for _b in _orden:
    if not np.isfinite(SALTO[_b]):
        continue
    if all(abs(LAM_BLOQUE[_b] - LAM_BLOQUE[_c]) > 2.5 * SNAP_A
           for _c in SALTOS_ELEGIDOS):
        SALTOS_ELEGIDOS.append(int(_b))
    if len(SALTOS_ELEGIDOS) == 2:
        break

_yy, _xx = np.mgrid[:CUBE.shape[1], :CUBE.shape[2]]
_rr = np.hypot(_yy - STAR_YX[0], _xx - STAR_YX[1])
DENTRO = _rr <= R_NORM

def _circulos(ax):
    for _c, _r, _col, _ls in ((STAR_YX, R_NORM, 'cyan', '-'),
                              (STAR_YX, STAR_RADIUS_PX, 'w', '--'),
                              (COMP_YX, COMP_RADIUS_PX, 'lime', '--')):
        ax.add_patch(plt.Circle((_c[1], _c[0]), _r, fill=False,
                                ec=_col, lw=0.8, ls=_ls))

def _log(ax, img, titulo):
    _pos = img[np.isfinite(img) & (img > 0)]
    _v0 = float(np.nanpercentile(_pos, 30)) if _pos.size else 1e-3
    ax.imshow(np.clip(img, _v0, None), origin='lower', cmap='inferno',
              norm=LogNorm(vmin=_v0,
                           vmax=float(np.nanpercentile(_pos, 99.9))))
    ax.set_title(titulo, fontsize=7.5); ax.set_xticks([]); ax.set_yticks([])
    _circulos(ax)

def _div(ax, img, titulo):
    _v = float(np.nanpercentile(np.abs(img[np.isfinite(img)]), 99))
    _im = ax.imshow(img, origin='lower', cmap='RdBu_r', vmin=-_v, vmax=_v)
    ax.set_title(titulo, fontsize=7.5); ax.set_xticks([]); ax.set_yticks([])
    _circulos(ax); return _im

for _k, _b in enumerate(SALTOS_ELEGIDOS, 1):
    _ma = (BLOQUE == _b - 1) & VALIDO
    _mb = (BLOQUE == _b) & VALIDO
    _lam = (float(np.nanmean(WAVE[_ma])), float(np.nanmean(WAVE[_mb])))
    _D = [np.nanmedian(CUBE[_ma], axis=0), np.nanmedian(CUBE[_mb], axis=0)]
    _M = [np.nanmedian(AJUSTE.model_cube[_ma], axis=0),
          np.nanmedian(AJUSTE.model_cube[_mb], axis=0)]
    _R = [np.nanmedian(AJUSTE.residual_cube[_ma], axis=0),
          np.nanmedian(AJUSTE.residual_cube[_mb], axis=0)]
    print(f'salto {_k}: w_eff {LAM_BLOQUE[_b - 1]:.0f} -> {LAM_BLOQUE[_b]:.0f} Å'
          f' · cociente {MED_BLOQUE[_b - 1]:.4f} -> {MED_BLOQUE[_b]:.4f}'
          f' ({100 * SALTO[_b]:+.1f}%)')
    for _et, _m in (('antes  ', _ma), ('después', _mb)):
        print(f'   {_et}: primaria {np.nanmedian(AJUSTE.coeffs[_m, 0]):10.4g}'
              f' · compañero {np.nanmedian(AJUSTE.coeffs[_m, 1]):8.4g}'
              f' · plano {np.nanmedian(AJUSTE.coeffs[_m, 2]):7.3g}'
              f' · χ²ᵣ {np.nanmedian(AJUSTE.chi2r[_m]):7.1f}')
    print('   la apertura, a los mismos dos lados:'
          f' {np.nanmedian(F_NORM[_ma]):.4g} -> {np.nanmedian(F_NORM[_mb]):.4g}')

    fig, axes = plt.subplots(3, 4, figsize=(12.5, 9.6))
    for _f in (0, 1):
        _et = 'antes' if _f == 0 else 'después'
        _log(axes[_f, 0], _D[_f], f'{_et} · dato  λ={_lam[_f]:.1f} Å')
        _log(axes[_f, 1], np.where(DENTRO, _D[_f], np.nan),
             f'{_et} · lo que suma la apertura r={R_NORM:.0f}')
        _log(axes[_f, 2], _M[_f], f'{_et} · modelo del psffit')
        _im = _div(axes[_f, 3], _R[_f], f'{_et} · residuo (dato − modelo)')
        fig.colorbar(_im, ax=axes[_f, 3], fraction=0.046)
    axes[2, 1].axis('off')
    for _c, _par, _t in ((0, (_D[0], _D[1]), 'dato'),
                         (2, (_M[0], _M[1]), 'modelo'),
                         (3, (_R[0], _R[1]), 'residuo')):
        _im = _div(axes[2, _c], _par[1] - _par[0], f'{_t}: después − antes')
        fig.colorbar(_im, ax=axes[2, _c], fraction=0.046)
    fig.suptitle(f'salto {_k} · w_eff {LAM_BLOQUE[_b - 1]:.0f} → '
                 f'{LAM_BLOQUE[_b]:.0f} Å · compara las ESCALAS de la fila 3:'
                 ' cuánto cambia el dato y cuánto el modelo', fontsize=9.5)
    fig.tight_layout(); plt.show()

    _bins = np.arange(0, 40, 1.0)
    _idx = np.digitize(_rr.ravel(), _bins) - 1
    def _perfil(img):
        _v = img.ravel(); _out = np.full(_bins.size - 1, np.nan)
        for _j in range(_out.size):
            _sel = (_idx == _j) & np.isfinite(_v)
            if _sel.sum() >= 3:
                _out[_j] = float(np.nanmedian(_v[_sel]))
        return _out
    _rc = 0.5 * (_bins[1:] + _bins[:-1])
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.6, 5.8), sharex=True,
                                   gridspec_kw={'height_ratios': [2, 1]})
    for _f, _col, _et in ((0, 'tab:blue', 'antes'), (1, 'tab:red', 'después')):
        ax1.plot(_rc, _perfil(_D[_f]), color=_col, lw=1.0, label=f'dato {_et}')
        ax1.plot(_rc, _perfil(_M[_f]), color=_col, lw=1.0, ls='--',
                 label=f'modelo {_et}')
        with np.errstate(invalid='ignore', divide='ignore'):
            ax2.plot(_rc, _perfil(_M[_f]) / _perfil(_D[_f]), color=_col, lw=1.0,
                     label=_et)
    ax1.set_yscale('log'); ax1.legend(fontsize=7); ax1.set_ylabel('perfil radial')
    ax1.set_title(f'salto {_k} · el perfil radial, antes y después', fontsize=9)
    ax2.axhline(1.0, color='0.6', lw=0.7, ls=':')
    ax2.axvline(R_NORM, color='cyan', lw=0.8)
    ax2.axvline(STAR_RADIUS_PX, color='0.4', lw=0.8, ls='--')
    ax2.set_ylim(0, 2); ax2.set_ylabel('modelo / dato', fontsize=8)
    ax2.set_xlabel('r [px]'); ax2.legend(fontsize=7)
    fig.tight_layout(); plt.show()


## 16 · Tres maneras de elegir la λ del modelo

Mismo cubo, mismo ajuste, **a resolución completa** en una ventana estrecha, cambiando solo `psfao_wave_bin_A`:

| perilla | qué hace |
|---|---|
| `50` | la cadena: redondea a 50 Å e interpola `param_table` |
| `0` | λ exacta, **sin** redondeo — sigue interpolando |
| `100` | cae sobre la rejilla de bins de C1 (desfase 0.53 Å): **casi no interpola** |

Separa las dos hipótesis: si el culpable fuera el redondeo, `0` saldría plano. Si es la interpolación, `0` se hunde igual —solo que suave, y con los máximos encima de los bins— y el que sale plano es `100`.


In [ ]:
VENTANA_SALTOS_A = (7150.0, 7400.0)   # perilla: ventana a resolución completa
PASO_EXACTA = 3                       # perilla: la λ exacta paga una FFT por canal

_sel = ((WAVE_FULL >= VENTANA_SALTOS_A[0]) & (WAVE_FULL < VENTANA_SALTOS_A[1]))
IDX_VENTANA = np.where(_sel)[0]

def _cociente_con(idx, wave_bin):
    """psffit/apertura sobre `idx`, con la PSF evaluada a `wave_bin`."""
    modelo = dict(PSF_MODEL)          # solo se añade una clave de primer nivel
    modelo['psfao_wave_bin_A'] = float(wave_bin)
    _var = (estimate_variance_cube(CUBE_FULL[idx]) if STAT_FULL is None
            else np.asarray(STAT_FULL[idx], dtype=float)
            * float(X03.get('x03_stat_factor_spaxel', 1.0) or 1.0))
    _aj = fit_psffit_cube(CUBE_FULL[idx], _var, WAVE_FULL[idx], STAR_YX, COMP_YX,
                          modelo, star_radius_px=STAR_RADIUS_PX,
                          comp_radius_px=COMP_RADIUS_PX, n_jobs=1)
    _ap = extract_aperture_spectrum(CUBE_FULL[idx], STAR_YX, float(R_NORM))
    with np.errstate(invalid='ignore', divide='ignore'):
        return _aj.coeffs[:, 0] / _ap

_variantes = [('snap 50 Å (la cadena)', IDX_VENTANA, 50.0, 'tab:red'),
              ('λ exacta, sin snap', IDX_VENTANA[::PASO_EXACTA], 0.0, 'k'),
              ('snap 100 Å (en los bins)', IDX_VENTANA, 100.0, 'tab:blue')]
fig, ax = plt.subplots(figsize=(11.5, 3.8))
for _n in NODOS:
    if VENTANA_SALTOS_A[0] <= _n <= VENTANA_SALTOS_A[1]:
        ax.axvline(_n, color='0.85', lw=0.8, zorder=0)
for _etq, _idx, _wb, _col in _variantes:
    _r = _cociente_con(_idx, _wb)
    _f6 = np.isfinite(_r)
    ax.plot(WAVE_FULL[_idx], _r, lw=0.9, color=_col, label=_etq)
    _disp = float(np.nanpercentile(_r[_f6], 98) - np.nanpercentile(_r[_f6], 2))
    print(f'{_etq:26s} mediana {float(np.nanmedian(_r)):.4f}'
          f' · dispersión p2–p98 {_disp:.4f}')
ax.set_xlabel('λ [Å]'); ax.set_ylabel(f'psffit / apertura r={R_NORM:.0f}', fontsize=8)
ax.set_title('§16 · gris = los bins de C1. La λ exacta se hunde igual entre bins:'
             ' el redondeo no es la causa, la interpolación sí', fontsize=9)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 17 · Las dos rejillas sobre todo el rango

El mismo ajuste de la §8, canal por canal, con la **otra** rejilla: 100 Å si la cadena va a 50, y 50 Å si el modelo ya lleva 100 escrito. El A/B vale en los dos sentidos; solo cambia qué columna es la buena.

Evaluar en los bins no es una solución completa —en los huecos de `param_table` sigue interpolando a ciegas, y la **pendiente** en λ no la toca— pero mide cuánto de la deformación era sólo esto.

Lo que hay que mirar es la **dispersión** (p2–p98), no la mediana.


In [ ]:
# Perilla del experimento. Por defecto, LA OTRA rejilla: si la cadena
# ya evalúa en 100 (el modelo lo lleva escrito), comparar contra 100
# sería un no-op y la sección no diría nada. Así el A/B se mantiene en
# los dos sentidos, y con 100 puesto el signo se invierte: lo que sale
# peor es la columna de la izquierda.
SNAP_NUEVO_A = 50.0 if SNAP_A >= 100.0 else 100.0

_modelo_nuevo = dict(PSF_MODEL)
_modelo_nuevo['psfao_wave_bin_A'] = float(SNAP_NUEVO_A)
AJUSTE_NUEVO = fit_psffit_cube(CUBE, VARIANZA, WAVE, STAR_YX, COMP_YX, _modelo_nuevo,
                               star_radius_px=STAR_RADIUS_PX,
                               comp_radius_px=COMP_RADIUS_PX, n_jobs=1)
with np.errstate(invalid='ignore', divide='ignore'):
    RATIO_NUEVO = AJUSTE_NUEVO.coeffs[:, 0] / F_NORM

def _p2p98(valores, mascara):
    _v = valores[mascara & np.isfinite(valores)]
    if _v.size < 20:
        return np.nan
    return float(np.nanpercentile(_v, 98) - np.nanpercentile(_v, 2))

print(f'χ²ᵣ mediano: {SNAP_A:.0f} Å {float(np.nanmedian(AJUSTE.chi2r)):8.1f}'
      f'   ->  {SNAP_NUEVO_A:.0f} Å {float(np.nanmedian(AJUSTE_NUEVO.chi2r)):8.1f}')
print(f'\nbanda [Å]        mediana {SNAP_A:.0f}/{SNAP_NUEVO_A:.0f}'
      f'        dispersión p2–p98 {SNAP_A:.0f}/{SNAP_NUEVO_A:.0f}')
for _lo, _hi in ((4800, 5700), (6100, 7000), (7000, 8000), (8000, 9350)):
    _m = VALIDO & (WAVE >= _lo) & (WAVE < _hi)
    if _m.sum() < 20:
        continue
    _d0, _d1 = _p2p98(RATIO_PSF, _m), _p2p98(RATIO_NUEVO, _m)
    print(f'{_lo}–{_hi}    {float(np.nanmedian(RATIO_PSF[_m])):.4f} /'
          f' {float(np.nanmedian(RATIO_NUEVO[_m])):.4f}'
          f'      {_d0:.4f} / {_d1:.4f}   (×{_d0 / _d1:.1f})')
# El compañero es débil y su cociente es ruidoso, así que la mediana
# sobre TODO el rango no dice nada (los canales que caen encima de un
# bin no se mueven, y son la mitad). Lo que se mira es dónde se mueve.
with np.errstate(invalid='ignore', divide='ignore'):
    _comp = AJUSTE.coeffs[:, 1] / AJUSTE_NUEVO.coeffs[:, 1]
_entre = (VALIDO & (DIST_NODO >= 25.0)) if NODOS.size else VALIDO
_cf = np.isfinite(_comp)
print(f'\ncompañero, cociente {SNAP_A:.0f}/{SNAP_NUEVO_A:.0f} Å:'
      f' mediana {100 * (float(np.nanmedian(_comp[VALIDO & _cf])) - 1):+.1f}%'
      ' en todo el rango,'
      f' {100 * (float(np.nanmedian(_comp[_entre & _cf])) - 1):+.1f}%'
      ' donde w_eff cae entre bins')
print('   (la dispersión canal a canal de ese cociente NO se cita: sobre una'
      ' fuente tan débil está dominada por el ruido, no por el escalón.'
      ' Lo que sí es del escalón es la mediana de la derecha, y los dos'
      ' bloques de la §15, donde el compañero se mueve de largo.)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.5, 5.6), sharex=True)
for _n in NODOS:
    for _ax in (ax1, ax2):
        _ax.axvline(_n, color='0.88', lw=0.5, zorder=0)
ax1.plot(WAVE[VALIDO], RATIO_PSF[VALIDO], lw=0.6, color='tab:red',
         label=f'snap {SNAP_A:.0f} Å (la cadena)')
ax1.plot(WAVE[VALIDO], RATIO_NUEVO[VALIDO], lw=0.6, color='tab:blue',
         label=f'snap {SNAP_NUEVO_A:.0f} Å')
ax1.set_ylabel(f'psffit / apertura r={R_NORM:.0f}', fontsize=8)
ax1.legend(fontsize=8)
ax1.set_title('§17 · los escalones son de la rejilla; la PENDIENTE en λ no:'
              ' sobrevive a las dos', fontsize=9)
with np.errstate(invalid='ignore', divide='ignore'):
    _coc17 = RATIO_NUEVO / RATIO_PSF
ax2.plot(WAVE[VALIDO], _coc17[VALIDO], lw=0.6, color='tab:purple')
ax2.axhline(1.0, color='0.6', lw=0.7, ls=':')
ax2.set_ylabel(f'{SNAP_NUEVO_A:.0f} Å / {SNAP_A:.0f} Å', fontsize=8)
ax2.set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 18 · Comparación con la cadena

Con las perillas por defecto, lo reconstruido aquí tiene que ser **exactamente** el producto: si no, esta cascada no describe la cadena y nada de lo de arriba vale.


In [ ]:
def compara(nombre, mio, fichero, rtol=1e-9):
    ref = SpectrumProduct.read(SD / fichero)
    suyo = np.asarray(ref.flux, float)[CANALES]
    _fin4 = np.isfinite(mio) & np.isfinite(suyo)
    _ig = np.isclose(mio[_fin4], suyo[_fin4], rtol=rtol, atol=0.0)
    print(f'{nombre} vs {fichero}: idénticos {100 * _ig.mean():6.2f}%'
          f' de {_fin4.sum()} canales | máx |Δ| ='
          f' {float(np.max(np.abs(mio - suyo)[_fin4])):.3e}')
    return bool(_ig.all())

_ok = compara('primaria C4  ', F_TOTAL, 'spec_psffit_star.fits')
_ok &= compara('primaria D2  ', F_TOTAL, 'spec_calibrated_psffit_star.fits')
if not _ok and GROWTH is not None:
    # Antes de culpar a la copia: el producto en disco puede ser
    # ANTERIOR al cambio de interpolador (2026-08-11), y entonces lleva
    # la parábola. Se comprueba en vez de suponerlo.
    _apc_prod = np.asarray(C4_PROD.apcorr, float)[CANALES]
    _cual = {}
    for _m2 in ('pchip', 'poly2'):
        _f5 = np.asarray(factor_at_wavelengths(GROWTH, WAVE, method=_m2,
                                              require_monotonic=False), float)
        _cual[_m2] = float(np.nanmax(np.abs(_apc_prod / _f5 - 1)))
    _lleva = min(_cual, key=_cual.get)
    print(f'\nel producto en disco lleva `{_lleva}`'
          f' (|Δ| máx {100 * _cual[_lleva]:.3f}%)')
    if _lleva != 'pchip':
        print('  -> es anterior al cambio del 2026-08-11: la diferencia de arriba'
              ' es ESO, no un fallo de la copia.')
print()
print('IDÉNTICO: la copia reproduce la cadena.' if _ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, mira la deriva.')


## 19 · Qué NO decide este notebook

1. **No toca `musepipe`.** Es un notebook de análisis: enseña la cascada y deja los dos números enfrentados.
2. **No dice que la apcorr sea el único culpable.** Separa lo que es del modelo de PSF (§12) de lo que es del factor empírico (§13); cerrar el 19 % puede exigir tocar los dos.
3. **No mide el flujo total de verdad.** `R_GRANDE` sigue dejando halo fuera: lo que se compara son **formas en λ**, no escalas absolutas.
4. **No cambia `psfao_wave_bin_A` en la cadena.** Las §§14–17 miden el escalón y enseñan que con `100` desaparece, pero esa perilla vive en `psf_model.json` (no hay knob en el config) y tocarla mueve **todos** los productos que consumen la PSF de C1 —C2, C3 `optimal_psfsub`, C4, la apcorr de E1—. Es decisión científica, y aquí solo está el número.
5. **No separa los dos defectos más allá de medirlos.** Son dos apilados: el **escalón** de las §§14–17 (interpolar entre bins) y la **pendiente** de la §12 (el halo del modelo es poco cromático). El `100` borra el primero y deja el segundo intacto.
